# 03 Torch Model Training - Data Smoke Test

This notebook is the first PyTorch baby step. It does not train a model. It only verifies that PyTorch can read the existing NB02 split manifests and produce the same target tensors used by TensorFlow NB03.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GLOBAL_SEED = 37
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('GLOBAL_SEED:', GLOBAL_SEED)


PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN
GLOBAL_SEED: 37


In [2]:
from src_torch.config import EXPERIMENT_CONFIG, SPLIT_DIR, TORCH_DATA_CONFIG
from src_torch.data import (
    GreenSpaceTorchDataset,
    load_split_dfs,
    make_dataloader,
    make_eval_dataloader,
    make_train_dataloader,
    missing_image_paths,
    resolve_split_schema,
    tensor_shapes,
    torch_available,
    validate_split_df,
)
from src_torch.sampling import oversampling_sanity_check
from src_torch.transforms import build_image_transform, tensor_summary

print('SPLIT_DIR:', SPLIT_DIR)
print('TORCH_DATA_CONFIG:', TORCH_DATA_CONFIG)
print('exclude_binary:', EXPERIMENT_CONFIG['exclude_binary'])
print('torch_available:', torch_available())


SPLIT_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/data/processed/splits
TORCH_DATA_CONFIG: {'img_size': (512, 512), 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'image_transform': 'tf_parity', 'backbone_preprocess': None, 'use_oversampling': True, 'use_augmentation': True, 'oversampling_sanity_batches': 80}
exclude_binary: ['gardens_p']
torch_available: True


In [3]:
split_dfs = load_split_dfs()
for split, df in split_dfs.items():
    schema = resolve_split_schema(df)
    validate_split_df(df, schema)
    missing = missing_image_paths(df, limit=3)
    print(f'{split}: rows={len(df)}, binary_cols={schema.binary_cols}')
    print(f'{split}: missing_image_path_examples={missing}')
    score_range = (float(df['score_mean'].min()), float(df['score_mean'].max()))
    veg_range = (float(df['veg_mean'].min()), float(df['veg_mean'].max()))
    print(f'{split}: full score_mean range={score_range}, full veg_mean range={veg_range}')


train: rows=3067, binary_cols=['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']
train: missing_image_path_examples=[]
train: full score_mean range=(1.0, 5.0), full veg_mean range=(1.0, 5.0)
val: rows=1022, binary_cols=['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']
val: missing_image_path_examples=[]
val: full score_mean range=(1.0, 5.0), full veg_mean range=(1.0, 5.0)
test: rows=1022, binary_cols=['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']
test: missing_image_path_examples=[]
test: full score_mean range=(1.0, 5.0), full veg_mean range=(1.0, 5.0)


The next cells require PyTorch. If `torch_available` above is `False`, install the repo `requirements.txt` before continuing.

In [4]:
train_schema = resolve_split_schema(split_dfs['train'])
train_ds = GreenSpaceTorchDataset(split_dfs['train'], schema=train_schema)
image, targets = train_ds[0]

print('sample shapes:', tensor_shapes(image, targets))
print('sample image summary:', tensor_summary(image))
print('score target:', targets['score_head'].tolist())
print('veg target:', targets['veg_head'].tolist())


sample shapes: {'image': (3, 512, 512), 'bin_head': (7,), 'shade_head': (), 'score_head': (1,), 'veg_head': (1,)}
sample image summary: {'shape': (3, 512, 512), 'dtype': 'torch.float32', 'min': 0.0, 'max': 0.9254902005195618}
score target: [1.0]
veg target: [3.0]


In [5]:
train_loader = make_dataloader('train', batch_size=TORCH_DATA_CONFIG['batch_size'], shuffle=False)
batch_images, batch_targets = next(iter(train_loader))

print('batch shapes:', tensor_shapes(batch_images, batch_targets))
print('batch image summary:', tensor_summary(batch_images))
print('batch score range:', float(batch_targets['score_head'].min()), float(batch_targets['score_head'].max()))
print('batch veg range:', float(batch_targets['veg_head'].min()), float(batch_targets['veg_head'].max()))


batch shapes: {'image': (4, 3, 512, 512), 'bin_head': (4, 7), 'shade_head': (4,), 'score_head': (4, 1), 'veg_head': (4, 1)}
batch image summary: {'shape': (4, 3, 512, 512), 'dtype': 'torch.float32', 'min': 0.0, 'max': 0.9607843160629272}
batch score range: 1.0 5.0
batch veg range: 2.0 4.0


Oversampling and augmentation parity smoke check: train uses the TensorFlow-aligned oversampling streams and augmentation; val/test stay plain.


In [6]:
from PIL import Image

train_parity_loader, oversampling_plan = make_train_dataloader(
    batch_size=TORCH_DATA_CONFIG['batch_size'],
    image_transform='rgb_255',
    use_oversampling=TORCH_DATA_CONFIG['use_oversampling'],
    use_augmentation=TORCH_DATA_CONFIG['use_augmentation'],
    seed=GLOBAL_SEED,
    return_plan=True,
)
val_plain_loader = make_eval_dataloader(
    'val',
    batch_size=TORCH_DATA_CONFIG['batch_size'],
    image_transform='rgb_255',
)

print('oversampling plan:', oversampling_plan.summary() if oversampling_plan else None)
parity_images, parity_targets = next(iter(train_parity_loader))
val_images, val_targets = next(iter(val_plain_loader))

print('train parity batch shapes:', tensor_shapes(parity_images, parity_targets))
print('train parity image summary:', tensor_summary(parity_images))
print('val plain batch shapes:', tensor_shapes(val_images, val_targets))
print('val plain image summary:', tensor_summary(val_images))

sanity = oversampling_sanity_check(
    train_parity_loader,
    train_schema.binary_cols,
    max_batches=TORCH_DATA_CONFIG['oversampling_sanity_batches'],
)
print('oversampling sanity:', sanity)

first_path = split_dfs['train'].iloc[0]['image_path']
plain_transform = build_image_transform(mode='rgb_255', augment=False)
aug_transform = build_image_transform(mode='rgb_255', augment=True)
with Image.open(first_path) as img:
    plain_image = plain_transform(img)
with Image.open(first_path) as img:
    augmented_a = aug_transform(img)
with Image.open(first_path) as img:
    augmented_b = aug_transform(img)

print('plain image summary:', tensor_summary(plain_image))
print('augmented image summary A:', tensor_summary(augmented_a))
print('augmented image summary B:', tensor_summary(augmented_b))
print('augmented mean abs delta A_vs_B:', float((augmented_a - augmented_b).abs().mean()))


oversampling plan: {'active': True, 'streams': [{'label': 'children_s_playground_p', 'target_rate': 0.2, 'pos_threshold': 0.5, 'row_count': 381, 'base_soft_mean': 0.1127757852407347, 'threshold_rate': 0.12422562764916857}, {'label': 'water_feature_p', 'target_rate': 0.25, 'pos_threshold': 0.5, 'row_count': 557, 'base_soft_mean': 0.18941419410933596, 'threshold_rate': 0.20052168242582327}], 'remainder_count': 2129, 'remainder_weight': 0.55, 'weights': [0.2, 0.25, 0.55]}
train parity batch shapes: {'image': (4, 3, 512, 512), 'bin_head': (4, 7), 'shade_head': (4,), 'score_head': (4, 1), 'veg_head': (4, 1)}
train parity image summary: {'shape': (4, 3, 512, 512), 'dtype': 'torch.float32', 'min': 0.0, 'max': 255.0}
val plain batch shapes: {'image': (4, 3, 512, 512), 'bin_head': (4, 7), 'shade_head': (4,), 'score_head': (4, 1), 'veg_head': (4, 1)}
val plain image summary: {'shape': (4, 3, 512, 512), 'dtype': 'torch.float32', 'min': 0.0, 'max': 251.0}
oversampling sanity: {'samples': 320, 'rat

Forward-pass smoke test only: build the selected TorchGeo ResNet-50 wrapper, pass one raw-RGB batch through it, and inspect output shapes. This does not train, compute losses, save checkpoints, or write run artifacts.

In [7]:
import torch

from src_torch.config import TORCH_MODEL_CONFIG
from src_torch.models import build_torchgeo_resnet50_forward_model, output_summary

model_loader = make_dataloader(
    'train',
    batch_size=TORCH_DATA_CONFIG['batch_size'],
    shuffle=False,
    image_transform='rgb_255',
)
model_images, model_targets = next(iter(model_loader))

model = build_torchgeo_resnet50_forward_model(
    load_pretrained_weights=TORCH_MODEL_CONFIG['load_pretrained_weights']
)
model.eval()

with torch.no_grad():
    preprocessed_images = model.preprocess_images(model_images)
    outputs = model(model_images)

print('model config:', TORCH_MODEL_CONFIG)
print('model metadata:', model.metadata())
print('raw model image summary:', tensor_summary(model_images))
print('preprocessed image summary:', tensor_summary(preprocessed_images))
print('target shapes:', tensor_shapes(model_images, model_targets))
print('output summary:', output_summary(outputs))


model config: {'backbone_priority': 'torchgeo', 'torchgeo_model_name': 'resnet50', 'torchgeo_weight': 'ResNet50_Weights.FMOW_RGB_GASSL', 'load_pretrained_weights': True, 'fallback_backbone': 'torchvision', 'num_binary': 7, 'num_shade': 2, 'score_output_range': (1.0, 5.0), 'veg_output_range': (1.0, 5.0)}
model metadata: {'model_name': 'resnet50', 'weight_name': 'ResNet50_Weights.FMOW_RGB_GASSL', 'load_pretrained_weights': True, 'weight_meta': {'dataset': 'fMoW Dataset', 'in_chans': 3, 'model': 'resnet50', 'publication': 'https://arxiv.org/abs/2011.09980', 'repo': 'https://github.com/sustainlab-group/geography-aware-ssl', 'ssl_method': 'gassl'}, 'weight_transforms': 'Sequential(\n  (0): Resize(size=[224, 224], interpolation=InterpolationMode.BILINEAR, antialias=True)\n  (1): Normalize(mean=[0], std=[255], inplace=True)\n  (2): Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=True)\n)'}
raw model image summary: {'shape': (4, 3, 512, 512), 'dtype': 'torch.float32', 

One-batch loss and metric smoke test only. This mirrors the active TensorFlow NB03 loss setup closely enough for wiring checks: binary BCE logits, shade sparse CE weighted by `walking_paths_p`, score/VEG MAE, and equal loss weights. It does not compute PR-AUC because a single batch of four samples is not a meaningful PR-AUC evaluation.

In [8]:
from src_torch.config import TORCH_LOSS_WEIGHTS
from src_torch.losses import compute_one_batch_diagnostics

loss_diagnostics = compute_one_batch_diagnostics(
    outputs=outputs,
    targets=model_targets,
    binary_cols=train_schema.binary_cols,
    loss_weights=TORCH_LOSS_WEIGHTS,
)

print('loss weights:', TORCH_LOSS_WEIGHTS)
print('one-batch diagnostics:', loss_diagnostics)


loss weights: {'bin_head': 1.0, 'shade_head': 1.0, 'score_head': 1.0, 'veg_head': 1.0}
one-batch diagnostics: {'loss_total': 3.165574312210083, 'loss_bin_head': 0.7003766298294067, 'loss_shade_head': 0.42678895592689514, 'loss_score_head': 1.2619023323059082, 'loss_veg_head': 0.7765064835548401, 'metric_bin_binary_accuracy': 0.3571428656578064, 'metric_shade_sparse_categorical_accuracy_weighted': 1.0, 'metric_score_mae': 1.2619023323059082, 'metric_veg_mae': 0.7765064835548401, 'metric_score_veg_mae_mean': 1.0192043781280518, 'shade_sample_weight_mean': 0.625, 'shade_sample_weight_sum': 2.5}


Heads-only training smoke: freeze the TorchGeo backbone, train only the GreenSpace heads, and run backward/optimizer/debug/mini-epoch checks. This is print-only and writes no artifacts.

In [9]:
from torch.utils.data import Subset

from src_torch.config import TORCH_TRAINING_SMOKE_CONFIG
from src_torch.data import make_dataset
from src_torch.training import (
    backward_smoke_step,
    make_optimizer,
    optimizer_smoke_step,
    run_repeated_batch_debug,
    run_tiny_train_val_epoch,
    set_backbone_trainable,
    set_torch_seed,
    trainable_parameter_summary,
)

set_torch_seed(TORCH_TRAINING_SMOKE_CONFIG['seed'])

train_subset = Subset(
    make_dataset('train', image_transform='rgb_255'),
    range(TORCH_TRAINING_SMOKE_CONFIG['subset_size']),
)
val_subset = Subset(
    make_dataset('val', image_transform='rgb_255'),
    range(TORCH_TRAINING_SMOKE_CONFIG['subset_size']),
)

train_smoke_loader = torch.utils.data.DataLoader(
    train_subset,
    batch_size=TORCH_TRAINING_SMOKE_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TORCH_DATA_CONFIG['num_workers'],
    pin_memory=TORCH_DATA_CONFIG['pin_memory'],
)
val_smoke_loader = torch.utils.data.DataLoader(
    val_subset,
    batch_size=TORCH_TRAINING_SMOKE_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TORCH_DATA_CONFIG['num_workers'],
    pin_memory=TORCH_DATA_CONFIG['pin_memory'],
)

training_model = build_torchgeo_resnet50_forward_model(load_pretrained_weights=True)
set_backbone_trainable(training_model, TORCH_TRAINING_SMOKE_CONFIG['train_backbone'])
optimizer = make_optimizer(training_model, lr=TORCH_TRAINING_SMOKE_CONFIG['learning_rate'])
first_batch = next(iter(train_smoke_loader))

print('training smoke config:', TORCH_TRAINING_SMOKE_CONFIG)
print('trainable parameter summary:', trainable_parameter_summary(training_model))

backward_diagnostics = backward_smoke_step(training_model, first_batch, train_schema.binary_cols)
print('backward smoke diagnostics:', backward_diagnostics)

optimizer_diagnostics = optimizer_smoke_step(training_model, first_batch, train_schema.binary_cols, optimizer)
print('optimizer smoke diagnostics:', optimizer_diagnostics)


training smoke config: {'seed': 37, 'subset_size': 64, 'debug_steps': 20, 'batch_size': 4, 'learning_rate': 0.001, 'train_backbone': False}
trainable parameter summary: {'total_params': 23530571, 'trainable_params': 22539, 'frozen_params': 23508032, 'trainable_groups': ['bin_head', 'shade_head', 'score_head_raw', 'veg_head_raw'], 'groups': {'backbone': {'total': 23508032, 'trainable': 0}, 'bin_head': {'total': 14343, 'trainable': 14343}, 'shade_head': {'total': 4098, 'trainable': 4098}, 'score_head_raw': {'total': 2049, 'trainable': 2049}, 'veg_head_raw': {'total': 2049, 'trainable': 2049}}}
backward smoke diagnostics: {'loss_total': 3.133547067642212, 'loss_bin_head': 0.6937236785888672, 'loss_shade_head': 0.4289373755455017, 'loss_score_head': 1.2693192958831787, 'loss_veg_head': 0.7415665984153748, 'metric_bin_binary_accuracy': 0.4285714328289032, 'metric_shade_sparse_categorical_accuracy_weighted': 0.6000000238418579, 'metric_score_mae': 1.2693192958831787, 'metric_veg_mae': 0.7415

In [10]:
debug_history = run_repeated_batch_debug(
    training_model,
    first_batch,
    train_schema.binary_cols,
    optimizer,
    steps=TORCH_TRAINING_SMOKE_CONFIG['debug_steps'],
)

print('repeated-batch first step:', debug_history[0])
print('repeated-batch last step:', debug_history[-1])
print('repeated-batch steps:', len(debug_history))


repeated-batch first step: {'step': 1, 'loss_total': 3.0219101905822754, 'loss_bin_head': 0.6695927977561951, 'loss_shade_head': 0.4042469263076782, 'loss_score_head': 1.2333083152770996, 'loss_veg_head': 0.7147623300552368, 'metric_bin_binary_accuracy': 0.7857142686843872, 'metric_shade_sparse_categorical_accuracy_weighted': 0.6000000238418579, 'metric_score_mae': 1.2333083152770996, 'metric_veg_mae': 0.7147623300552368, 'metric_score_veg_mae_mean': 0.9740353226661682, 'shade_sample_weight_mean': 0.625, 'shade_sample_weight_sum': 2.5, 'grad_param_tensors': 8, 'grad_all_finite': True, 'grad_norm': 1.449355959892273}
repeated-batch last step: {'step': 20, 'loss_total': 1.4584321975708008, 'loss_bin_head': 0.36267098784446716, 'loss_shade_head': 0.1418757438659668, 'loss_score_head': 0.7354979515075684, 'loss_veg_head': 0.21838751435279846, 'metric_bin_binary_accuracy': 0.9285714030265808, 'metric_shade_sparse_categorical_accuracy_weighted': 1.0, 'metric_score_mae': 0.7354979515075684, '

In [11]:
epoch_summary = run_tiny_train_val_epoch(
    training_model,
    train_smoke_loader,
    val_smoke_loader,
    train_schema.binary_cols,
    optimizer,
)

print('tiny train/val epoch summary:', epoch_summary)


tiny train/val epoch summary: {'train': {'loss_total': 3.1475313156843185, 'loss_bin_head': 0.5565251726657152, 'loss_shade_head': 0.4288271814584732, 'loss_score_head': 1.2365618273615837, 'loss_veg_head': 0.9256171323359013, 'metric_bin_binary_accuracy': 0.6897321380674839, 'metric_shade_sparse_categorical_accuracy_weighted': 0.6739583443850279, 'metric_score_mae': 1.2365618273615837, 'metric_veg_mae': 0.9256171323359013, 'metric_score_veg_mae_mean': 1.0810894798487425, 'shade_sample_weight_mean': 0.6901041679084301, 'shade_sample_weight_sum': 2.7604166716337204, 'grad_param_tensors': 8.0, 'grad_all_finite': 1.0, 'grad_norm': 1.8353205248713493}, 'val': {'loss_total': 3.029100641608238, 'loss_bin_head': 0.5661318302154541, 'loss_shade_head': 0.469057435169816, 'loss_score_head': 1.0821692124009132, 'loss_veg_head': 0.9117422141134739, 'metric_bin_binary_accuracy': 0.6919642873108387, 'metric_shade_sparse_categorical_accuracy_weighted': 0.4016369068995118, 'metric_score_mae': 1.082169